# Semantic Model Similarity

Collect model schemas, security definitions and direct report dependencies, persist the catalog, then calculate similarity and schema containment in the same notebook. Source models and reports are read-only; catalog and scoring outputs replace their snapshots in the attached Lakehouse.

Run this notebook interactively, then open **002_semantic_model_similarity_results.ipynb** attached to the same Lakehouse. Optional temporary access changes workspace role assignments for a dedicated security group only during collection, not during scoring.

## Prerequisites

Attach a Lakehouse and run in a Fabric PySpark notebook runtime with Semantic Link Labs, pandas, scikit-learn and SciPy. The runtime identity needs complete model metadata access, including role definitions, plus report read access. Security collection reads definitions, not members or user assignments; validate full visibility on a known protected model.

Temporary workspace access additionally requires a Fabric administrator with admin API authorization (`Tenant.ReadWrite.All`) who already belongs to the configured security group. The group must be dedicated to this process, with no permanent workspace assignments or overlapping runs, including runs using another Lakehouse. Member access applies to every group member. XMLA, capacity and licensing prerequisites still apply.

In [ ]:
# Install the Semantic Link Labs package if it is not already available.
%pip install -q semantic-link-labs pandas

In [ ]:
import pandas as pd
import sempy.fabric as fabric
from sempy.fabric.exceptions import FabricHTTPException
from sempy_labs import admin
from sempy_labs.tom import connect_semantic_model

## Configuration

Workspace/model filters use exact names, ignoring case. `REPORT_WORKSPACE_NAME` is independent: narrowing models does not narrow report discovery. `None` (or a blank string) leaves the respective scope unfiltered. Results replace snapshots in the default attached Lakehouse.

The next cell is tagged as the Fabric **parameters** cell, so pipeline, scheduler or REST API runs can override the scope filters, `ENABLE_BLOCKING`, `DUPLICATE_THRESHOLD`, `SIMILAR_THRESHOLD` and `CONTAINMENT_THRESHOLD`. Values are validated in the scoring parameters cell after any injected overrides.

Temporary access is off by default. To enable it, set `ENABLE_TEMPORARY_WORKSPACE_ACCESS = True` and `ACCESS_SECURITY_GROUP_ID` to the dedicated group's object UUID. All selected active regular workspaces, including report-only workspaces, receive a direct Member-add attempt without a pre-grant assignment lookup. With no workspace filters, this can cover the tenant's active regular workspaces.

The next helper cell defines access management and recovery without granting access. Grants occur only when collection starts. A normal workspace is added, read, removed in `finally`, and checked for removal before continuing. Failed or ambiguous adds are reconciled separately; cleanup failures stop the run.

Interrupted-run journals are kept under `Files/semantic-model-similarity/workspace-access/<group-id>/<run-id>/`. After confirming the old run and any pending requests have stopped, call `recover_workspace_access("<run-id>", confirm_stopped=True)` with the same group and Lakehouse configured, then restart collection. Do not remove journals to bypass recovery. `finally` cannot execute after a killed session, and this journal is not a distributed lock.

In [ ]:
# Fabric parameters cell: keep assignments only. Pipeline/API runs inject overrides after this cell.
# Blank strings are treated as None because notebook parameters cannot pass None.
WORKSPACE_NAME = None
MODEL_NAME = None
REPORT_WORKSPACE_NAME = None

ENABLE_TEMPORARY_WORKSPACE_ACCESS = False
ACCESS_SECURITY_GROUP_ID = None
ACCESS_READY_TIMEOUT_SECONDS = 300
ACCESS_POLL_INTERVAL_SECONDS = 15

# Blocking limits comparisons to model pairs that share at least one table or measure name.
# Disable to force full pairwise comparison (slower on large catalogs).
ENABLE_BLOCKING = True

# Combined-score tier thresholds.
DUPLICATE_THRESHOLD = 0.95
SIMILAR_THRESHOLD = 0.70

# Containment threshold. Containment is a second, directional score that answers a
# different question than similarity: "does one model contain the schema definitions
# of the other?" Security is not part of directional containment.
CONTAINMENT_THRESHOLD = 0.95


## Scoring Parameters

Set signal weights before collection; tier thresholds and blocking live in the parameters cell above, and this cell validates all parameters. Schema, security and combined similarity remain separate from directional schema containment. Defaults are unchanged; no external embedding endpoint is used.

In [ ]:
# Results are read from and written to the lakehouse attached to this notebook.
# ENABLE_BLOCKING and the tier/containment thresholds are set in the parameters cell above.

# Fabric notebook parameters cannot pass None, so blank filter names mean "unfiltered".
WORKSPACE_NAME, MODEL_NAME, REPORT_WORKSPACE_NAME = (
    None if isinstance(value, str) and not value.strip() else value
    for value in (WORKSPACE_NAME, MODEL_NAME, REPORT_WORKSPACE_NAME)
)
for filter_name, filter_value in (
    ("WORKSPACE_NAME", WORKSPACE_NAME),
    ("MODEL_NAME", MODEL_NAME),
    ("REPORT_WORKSPACE_NAME", REPORT_WORKSPACE_NAME),
):
    if filter_value is not None and not isinstance(filter_value, str):
        raise ValueError(f"{filter_name} must be None or a non-empty name.")
for flag_name, flag_value in (
    ("ENABLE_TEMPORARY_WORKSPACE_ACCESS", ENABLE_TEMPORARY_WORKSPACE_ACCESS),
    ("ENABLE_BLOCKING", ENABLE_BLOCKING),
):
    if not isinstance(flag_value, bool):
        raise ValueError(f"{flag_name} must be a boolean.")
for threshold_name, threshold_value in (
    ("DUPLICATE_THRESHOLD", DUPLICATE_THRESHOLD),
    ("SIMILAR_THRESHOLD", SIMILAR_THRESHOLD),
    ("CONTAINMENT_THRESHOLD", CONTAINMENT_THRESHOLD),
):
    if not isinstance(threshold_value, (int, float)) or isinstance(threshold_value, bool) or not 0 < threshold_value <= 1:
        raise ValueError(f"{threshold_name} must be a number greater than 0 and at most 1.")
if SIMILAR_THRESHOLD > DUPLICATE_THRESHOLD:
    raise ValueError("SIMILAR_THRESHOLD must not exceed DUPLICATE_THRESHOLD.")

# Report knobs.
TOP_N = 20  # Rows shown in the ranked pair table.
HEATMAP_MIN_SCORE = SIMILAR_THRESHOLD  # Hide heatmap cells scoring below this combined value.

# Relative weights for each schema similarity signal. Values are normalized.
# Power Query compares normalized M text; it is skipped for a pair when neither model has M.
SIMILARITY_WEIGHTS = {
    "tables": 0.15,
    "columns": 0.20,
    "measure_names": 0.10,
    "measure_dax_embedding": 0.20,
    "relationships": 0.10,
    "datasources": 0.10,
    "power_query": 0.15,
}

COMBINED_WEIGHTS = {"schema": 0.95, "security": 0.05}
SECURITY_WEIGHTS = {"role_definitions": 4.0, "rls_propagation": 1.0}
SECURITY_ROLE_WEIGHTS = {"model_permission": 1.0, "rls": 1.0, "table_ols": 1.0, "column_ols": 1.0}
SCORE_VERSION = 3

# Relative weights for each containment signal. Containment uses exact measure
# definitions (name + comment-stripped DAX) instead of the TF-IDF embedding, because
# lexical cosine similarity is symmetric and cannot establish that one model's measures
# are a subset of another's. Weights are normalized per direction over whichever signals
# the source model actually has.
CONTAINMENT_WEIGHTS = {
    "tables": 0.15,
    "columns": 0.20,
    "measure_names": 0.10,
    "measure_definitions": 0.20,
    "relationships": 0.10,
    "datasources": 0.10,
    "power_queries": 0.15,
}

# Guard against misconfigured weights before any scoring runs.
for _weights_name, _weights in (
    ("SIMILARITY_WEIGHTS", SIMILARITY_WEIGHTS),
    ("CONTAINMENT_WEIGHTS", CONTAINMENT_WEIGHTS),
    ("COMBINED_WEIGHTS", COMBINED_WEIGHTS),
    ("SECURITY_WEIGHTS", SECURITY_WEIGHTS),
    ("SECURITY_ROLE_WEIGHTS", SECURITY_ROLE_WEIGHTS),
):
    if any(not isinstance(weight, (int, float)) or isinstance(weight, bool) or not 0 <= weight < float("inf") for weight in _weights.values()):
        raise ValueError(f"{_weights_name} must contain finite nonnegative weights.")
    if sum(_weights.values()) <= 0:
        raise ValueError(f"{_weights_name} must sum to a positive value.")


In [ ]:
from contextlib import contextmanager
from datetime import datetime, timezone
from uuid import UUID
import inspect
import json
import math
import time


class WorkspaceAccessError(RuntimeError):
    pass


class WorkspaceGrantDenied(RuntimeError):
    pass


def access_http_status(error):
    response = getattr(error, "response", None)
    return getattr(response, "status_code", None) if response is not None else getattr(error, "status_code", None)


def access_is_transient(error, access=False):
    status = access_http_status(error)
    if status in {408, 429, 500, 502, 503, 504} | ({403, 404} if access else set()):
        return True
    if isinstance(error, (TimeoutError, ConnectionError)) or type(error).__name__ in {"ReadTimeout", "ConnectTimeout"}:
        return True
    if access and isinstance(error, PermissionError):
        return True
    if access and type(error).__name__ in {"ConnectionException", "AdomdConnectionException", "AdomdErrorResponseException"}:
        message = str(error).casefold()
        return any(text in message for text in ("permission", "unauthorized", "not authorized", "access denied"))
    return False


def access_retry_delay(error, interval):
    from email.utils import parsedate_to_datetime

    headers = getattr(getattr(error, "response", None), "headers", {}) or {}
    value = headers.get("Retry-After") or headers.get("retry-after")
    if value is None:
        return interval
    try:
        delay = float(value)
    except (ValueError, TypeError):
        try:
            delay = (parsedate_to_datetime(value) - datetime.now(timezone.utc)).total_seconds()
        except (ValueError, TypeError, OverflowError):
            return interval
    return max(interval, delay) if math.isfinite(delay) else interval


class LakehouseAccessJournal:
    def __init__(self, fs, group_id):
        self.fs = fs
        self.group_id = str(UUID(group_id))
        self.root = "Files/semantic-model-similarity/workspace-access/" + self.group_id

    def _path(self, run_id, name="run.json"):
        return self.root + "/" + str(UUID(run_id)) + "/" + name

    def _read(self, path):
        content = self.fs.head(path, 65537)
        if not isinstance(content, str) or len(content.encode("utf-8")) > 65536:
            raise WorkspaceAccessError("Access journal exceeds its size limit: " + path)
        document = json.loads(content)
        if not isinstance(document, dict) or document.get("version") != 1 or document.get("group_id") != self.group_id:
            raise WorkspaceAccessError("Invalid access journal: " + path)
        return document

    def _write(self, path, document):
        content = json.dumps(document, sort_keys=True, ensure_ascii=True, allow_nan=False)
        if len(content.encode("utf-8")) > 65536:
            raise WorkspaceAccessError("Access journal exceeds its size limit: " + path)
        if self.fs.put(path, content, True) is not True or self._read(path) != document:
            raise WorkspaceAccessError("Access journal write could not be verified: " + path)

    def read_run(self, run_id):
        run_id = str(UUID(run_id))
        manifest = self._read(self._path(run_id))
        if manifest.get("run_id") != run_id or manifest.get("kind") != "run" or manifest.get("state") not in {"active", "complete"}:
            raise WorkspaceAccessError("Invalid access run manifest: " + run_id)
        entries = []
        for item in self.fs.ls(self._path(run_id, "")):
            name = item.name.rstrip("/")
            if name == "run.json":
                continue
            if item.isDir or not name.endswith(".json"):
                raise WorkspaceAccessError("Unexpected access journal entry: " + name)
            workspace_id = str(UUID(name[:-5]))
            document = self._read(self._path(run_id, name))
            if (document.get("run_id") != run_id or document.get("workspace_id") != workspace_id
                    or document.get("kind") != "workspace" or document.get("role") != "Member"
                    or document.get("ownership") != "dedicated_group"
                    or document.get("state") not in {"intent", "granted", "recovered", "not_granted", "removed", "uncertain", "cleanup_failed"}
                    or document.get("origin") not in {"pending", "add_confirmed", "reconciled"}
                    or not isinstance(document.get("events"), list)):
                raise WorkspaceAccessError("Invalid workspace access journal: " + name)
            entries.append(document)
        return manifest, entries

    def begin(self, run_id, scope):
        if self.fs.exists(self.root):
            for item in self.fs.ls(self.root):
                if not item.isDir:
                    raise WorkspaceAccessError("Unexpected access journal file in " + self.root)
                previous_id = str(UUID(item.name.rstrip("/")))
                manifest, entries = self.read_run(previous_id)
                if manifest["state"] != "complete" or any(entry["state"] not in {"removed", "not_granted"} for entry in entries):
                    raise WorkspaceAccessError("Unfinished access run " + previous_id + "; use recover_workspace_access before starting another scan.")
        path = self._path(run_id, "")
        if self.fs.exists(path):
            raise WorkspaceAccessError("Access run already exists: " + run_id)
        if self.fs.mkdirs(path) is not True:
            raise WorkspaceAccessError("Cannot create access journal: " + path)
        self._write(self._path(run_id), {
            "version": 1, "kind": "run", "group_id": self.group_id, "run_id": str(UUID(run_id)),
            "state": "active", "created_at": datetime.now(timezone.utc).isoformat(),
            "workspace_count": len(scope["workspaces"]), "report_scope": scope["report_scope"],
        })

    def write_entry(self, run_id, entry):
        self._write(self._path(run_id, str(UUID(entry["workspace_id"])) + ".json"), entry)

    def finish(self, run_id):
        manifest, entries = self.read_run(run_id)
        if any(entry["state"] not in {"removed", "not_granted"} for entry in entries):
            raise WorkspaceAccessError("Access run still has unresolved workspace assignments: " + run_id)
        manifest.update(state="complete", completed_at=datetime.now(timezone.utc).isoformat())
        self._write(self._path(run_id), manifest)


class TemporaryWorkspaceAccess:
    def __init__(self, admin, group_id, journal, timeout=300, interval=15, clock=None, sleep=None):
        self.admin = admin
        self.group_id = str(UUID(group_id))
        if UUID(self.group_id).int == 0 or journal.group_id != self.group_id:
            raise ValueError("The nonempty access-group UUID must match its journal")
        for name, value in (("timeout", timeout), ("interval", interval)):
            if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:
                raise ValueError(name + " must be a finite positive number")
        if interval > timeout:
            raise ValueError("Access polling interval must not exceed the timeout")
        for function, arguments in (
            (admin.add_user_to_workspace, {"user": self.group_id, "role": "Member", "principal_type": "Group", "workspace": self.group_id}),
            (admin.delete_user_from_workspace, {"user": self.group_id, "workspace": self.group_id, "is_group": True}),
            (admin.list_workspace_users, {"workspace": self.group_id}),
        ):
            inspect.signature(function).bind(**arguments)
        self.journal = journal
        self.timeout = timeout
        self.interval = interval
        self.clock = clock or time.monotonic
        self.sleep = sleep or time.sleep
        self.run_id = None
        self.active_workspace = None
        self.workspace_ids = set()

    def begin(self, run_id, scope):
        if self.active_workspace is not None:
            raise WorkspaceAccessError("Workspace access must be sequential")
        try:
            self.workspace_ids = {str(UUID(workspace["workspace_id"])) for workspace in scope["workspaces"]}
            self.journal.begin(run_id, scope)
            self.run_id = str(UUID(run_id))
        except Exception as error:
            self.run_id = None
            raise WorkspaceAccessError("Access journal preflight failed; no new grants were made. " + str(error)) from error

    def finish(self):
        try:
            self.journal.finish(self.run_id)
            self.run_id = None
        except Exception as error:
            raise WorkspaceAccessError("Could not finish access journal for run " + str(self.run_id)) from error

    def _pause(self, deadline, error=None):
        delay = access_retry_delay(error, self.interval) if error is not None else self.interval
        if self.clock() + delay >= deadline:
            raise TimeoutError("Workspace access retry budget exhausted") from error
        self.sleep(delay)

    def _retry(self, operation, deadline, access=False):
        while True:
            try:
                return operation()
            except Exception as error:
                if not access_is_transient(error, access):
                    raise
                self._pause(deadline, error)

    def _role(self, workspace_id):
        frame = self.admin.list_workspace_users(workspace=workspace_id)
        if not {"Principal Type", "Group User Access Right"}.issubset(frame.columns) or not {"Graph Id", "Identifier"}.intersection(frame.columns):
            raise WorkspaceAccessError("Unrecognized workspace assignment response")
        matches = []
        for row in frame.to_dict("records"):
            principal_type = row.get("Principal Type")
            if not isinstance(principal_type, str):
                raise WorkspaceAccessError("Missing principal type in workspace assignment")
            if principal_type.casefold() != "group":
                continue
            identifiers = [value.casefold() for value in (row.get("Graph Id"), row.get("Identifier")) if isinstance(value, str) and value]
            if not identifiers:
                raise WorkspaceAccessError("Missing group identity in workspace assignment")
            if self.group_id in identifiers:
                role = row.get("Group User Access Right")
                if role not in {"Admin", "Member", "Contributor", "Viewer"}:
                    raise WorkspaceAccessError("Unrecognized group role")
                matches.append(role)
        if len(matches) > 1:
            raise WorkspaceAccessError("Multiple assignments for the access group")
        return matches[0] if matches else None

    def _save(self, entry, state, error=None):
        entry["state"] = state
        event = {"state": state, "at": datetime.now(timezone.utc).isoformat()}
        if error is not None:
            event.update(error_type=type(error).__name__, http_status=access_http_status(error))
        entry["events"].append(event)
        try:
            self.journal.write_entry(self.run_id, entry)
        except Exception as journal_error:
            raise WorkspaceAccessError("Could not persist access state for " + entry["workspace_id"] + "; run " + str(self.run_id)) from journal_error

    def _remove(self, workspace_id):
        deadline = self.clock() + self.timeout
        for attempt in range(3):
            try:
                self.admin.delete_user_from_workspace(user=self.group_id, workspace=workspace_id, is_group=True)
                break
            except Exception as error:
                role = self._retry(lambda: self._role(workspace_id), deadline)
                if role is None:
                    return
                if role != "Member":
                    raise WorkspaceAccessError("Unexpected access-group role during cleanup: " + role) from error
                if attempt == 2 or not access_is_transient(error):
                    raise
                self._pause(deadline, error)
        while True:
            role = self._retry(lambda: self._role(workspace_id), deadline)
            if role is None:
                return
            if role != "Member":
                raise WorkspaceAccessError("Unexpected access-group role after removal: " + role)
            self._pause(deadline)

    @contextmanager
    def workspace(self, workspace):
        workspace_id = str(UUID(workspace["workspace_id"]))
        if self.run_id is None or self.active_workspace is not None or workspace_id not in self.workspace_ids:
            raise WorkspaceAccessError("Workspace access must be initialized, in scope and sequential")
        entry = {
            "version": 1, "kind": "workspace", "run_id": self.run_id, "group_id": self.group_id,
            "workspace_id": workspace_id, "workspace_name": workspace["workspace_name"],
            "role": "Member", "ownership": "dedicated_group", "origin": "pending", "events": [],
        }
        self._save(entry, "intent")
        self.active_workspace = workspace_id
        cleanup_required = False
        original_error = None
        try:
            try:
                self.admin.add_user_to_workspace(user=self.group_id, role="Member", principal_type="Group", workspace=workspace_id)
            except BaseException as error:
                try:
                    role = self._retry(lambda: self._role(workspace_id), self.clock() + self.timeout)
                except BaseException as lookup_error:
                    self._save(entry, "uncertain", error)
                    raise WorkspaceAccessError("Add outcome could not be reconciled for " + workspace_id + "; run " + self.run_id) from lookup_error
                if role == "Member":
                    cleanup_required = True
                    entry["origin"] = "reconciled"
                    self._save(entry, "recovered", error)
                    if not isinstance(error, Exception):
                        raise
                    if access_http_status(error) in {401, 403, 429}:
                        raise WorkspaceAccessError("Grant authorization or rate limit failure; stop further grants") from error
                elif role is None and access_http_status(error) in {400, 401, 403, 404, 409, 422, 429}:
                    self._save(entry, "not_granted", error)
                    if access_http_status(error) in {401, 403, 429}:
                        raise WorkspaceAccessError("Grant authorization or rate limit failure; stop further grants") from error
                    raise WorkspaceGrantDenied("Could not grant access to " + workspace_id) from error
                else:
                    self._save(entry, "uncertain", error)
                    raise WorkspaceAccessError("Unresolved access-group assignment for " + workspace_id + "; run " + self.run_id) from error
            else:
                cleanup_required = True
                entry["origin"] = "add_confirmed"
                self._save(entry, "granted")
            deadline = self.clock() + self.timeout
            yield lambda operation: self._retry(operation, deadline, access=True)
        except BaseException as error:
            original_error = error
            raise
        finally:
            try:
                if cleanup_required:
                    if original_error is not None:
                        entry["scan_error_type"] = type(original_error).__name__
                    try:
                        self._remove(workspace_id)
                    except BaseException as cleanup_error:
                        try:
                            self._save(entry, "cleanup_failed", cleanup_error)
                        except Exception:
                            pass
                        detail = " after " + type(original_error).__name__ if original_error is not None else ""
                        raise WorkspaceAccessError(
                            "Cleanup failed" + detail + " for workspace " + workspace_id + ", group " + self.group_id
                            + "; stop new scans and recover run " + self.run_id
                        ) from cleanup_error
                    self._save(entry, "removed")
            finally:
                self.active_workspace = None

    def recover(self, run_id, confirm_no_pending_add=False):
        if confirm_no_pending_add is not True or self.active_workspace is not None:
            raise WorkspaceAccessError("Confirm the previous run and all pending requests have stopped before recovery")
        recovery_id = str(UUID(run_id))
        try:
            manifest, entries = self.journal.read_run(recovery_id)
            self.run_id = recovery_id
            for entry in entries:
                if entry["state"] in {"removed", "not_granted"}:
                    continue
                workspace_id = entry["workspace_id"]
                role = self._retry(lambda: self._role(workspace_id), self.clock() + self.timeout)
                if role not in {None, "Member"}:
                    raise WorkspaceAccessError("Unexpected recovery role for " + workspace_id + ": " + role)
                if role == "Member":
                    self._remove(workspace_id)
                self._save(entry, "removed")
            self.journal.finish(recovery_id)
        except Exception as error:
            raise WorkspaceAccessError("Recovery incomplete for run " + recovery_id + ": " + str(error)) from error
        finally:
            self.run_id = None
        print("Workspace access recovered for run " + recovery_id)


def configure_workspace_access(enabled, group_id, timeout, interval, admin, context=None, fs=None):
    if not isinstance(enabled, bool):
        raise ValueError("ENABLE_TEMPORARY_WORKSPACE_ACCESS must be a boolean")
    if not enabled:
        return None
    if not isinstance(group_id, str) or not group_id.strip():
        raise ValueError("Set ACCESS_SECURITY_GROUP_ID to the dedicated group's object UUID")
    group_id = str(UUID(group_id))
    if context is None or not context.get("defaultLakehouseId") or fs is None:
        raise ValueError("Attach a Lakehouse before enabling temporary workspace access")
    if context.get("isForPipeline") or context.get("isReferenceRun"):
        raise ValueError("Run this notebook interactively as a Fabric administrator who belongs to the access group")
    journal = LakehouseAccessJournal(fs, group_id)
    return TemporaryWorkspaceAccess(admin, group_id, journal, timeout, interval)


def recover_workspace_access(run_id, confirm_stopped=False):
    if workspace_access is None:
        raise WorkspaceAccessError("Enable and configure temporary access before recovery; do not run collection yet")
    workspace_access.recover(run_id, confirm_no_pending_add=confirm_stopped)


workspace_access = None
if ENABLE_TEMPORARY_WORKSPACE_ACCESS:
    import notebookutils
    workspace_access = configure_workspace_access(
        ENABLE_TEMPORARY_WORKSPACE_ACCESS, ACCESS_SECURITY_GROUP_ID, ACCESS_READY_TIMEOUT_SECONDS,
        ACCESS_POLL_INTERVAL_SECONDS, admin, notebookutils.runtime.context, notebookutils.fs,
    )


## Discover and Collect Workspaces

With temporary access enabled, discovery uses the admin API and selects only active regular workspaces. Otherwise, model discovery retains its admin/visible fallback and report discovery uses visible workspaces.

Each workspace is processed once for its selected model/report scopes. Models use read-only TOM connections; reports are linked to the complete model catalog after all workspace reads finish. Model errors do not skip report collection. Failed or partial security reads remain unknown, not evidence that security is absent.

In [ ]:
def collect_security_definition(model):
    import json

    result = {
        "security_schema_version": 1,
        "scan_status": "failed",
        "error_type": None,
        "role_count": None,
        "rls_filter_count": None,
        "table_ols_count": None,
        "column_ols_count": None,
        "definition_json": None,
    }
    roles = []

    def required_name(value):
        if value is None or not str(value).strip():
            raise ValueError("Missing security object name")
        return str(value)

    def required_enum(value, allowed):
        if value is None or str(value) not in allowed:
            raise ValueError("Unsupported security metadata value")
        return str(value)

    try:
        for role in model.Roles:
            role_definition = {
                "name": required_name(role.Name),
                "model_permission": required_enum(
                    role.ModelPermission, {"None", "Read", "ReadRefresh", "Refresh", "Administrator"}
                ),
                "tables": [],
            }
            for permission in role.TablePermissions:
                expression = permission.FilterExpression
                table_definition = {
                    "table": required_name(permission.Table.Name),
                    "filter_expression": str(expression) if expression is not None else "",
                    "metadata_permission": required_enum(permission.MetadataPermission, {"Default", "None", "Read"}),
                    "columns": [],
                }
                for column_permission in permission.ColumnPermissions:
                    table_definition["columns"].append({
                        "column": required_name(column_permission.Column.Name),
                        "metadata_permission": required_enum(column_permission.MetadataPermission, {"Default", "None", "Read"}),
                    })
                role_definition["tables"].append(table_definition)
            roles.append(role_definition)

        relationships = []
        for relationship in model.Relationships:
            active = relationship.IsActive
            if active is None:
                raise ValueError("Missing relationship active state")
            relationships.append({
                "from_table": required_name(relationship.FromTable.Name),
                "from_column": required_name(relationship.FromColumn.Name),
                "to_table": required_name(relationship.ToTable.Name),
                "to_column": required_name(relationship.ToColumn.Name),
                "is_active": bool(active),
                "cross_filtering_behavior": required_enum(
                    relationship.CrossFilteringBehavior, {"OneDirection", "BothDirections", "Automatic"}
                ),
                "security_filtering_behavior": required_enum(
                    relationship.SecurityFilteringBehavior, {"OneDirection", "BothDirections", "None"}
                ),
                "from_cardinality": required_enum(relationship.FromCardinality, {"One", "Many"}),
                "to_cardinality": required_enum(relationship.ToCardinality, {"One", "Many"}),
            })
        tables = [table for role in roles for table in role["tables"]]
        result.update({
            "scan_status": "complete",
            "role_count": len(roles),
            "rls_filter_count": sum(bool(table["filter_expression"].strip()) for table in tables),
            "table_ols_count": sum(table["metadata_permission"] == "None" for table in tables),
            "column_ols_count": sum(
                column["metadata_permission"] == "None" for table in tables for column in table["columns"]
            ),
            "definition_json": json.dumps({"roles": roles, "relationships": relationships}, sort_keys=True, ensure_ascii=True),
        })
    except Exception as error:
        result["scan_status"] = "partial" if roles else "failed"
        result["error_type"] = type(error).__name__
    return result


In [ ]:
def normalize_workspaces(frame, active_only=False):
    from uuid import UUID

    workspaces = {}
    for record in frame.to_dict("records"):
        if active_only and (str(record.get("State", "")).casefold() != "active"
                            or str(record.get("Type", "")).casefold() != "workspace"):
            continue
        workspace_id = next((record[key] for key in ("Id", "Workspace Id", "id") if key in record), None)
        name = next((record[key] for key in ("Name", "Workspace Name", "displayName") if key in record), None)
        if not isinstance(workspace_id, str) or not workspace_id.strip() or not isinstance(name, str) or not name.strip():
            raise ValueError("Missing workspace identity")
        workspace_id = str(UUID(workspace_id)) if active_only else workspace_id.strip().casefold()
        workspace = {"workspace_id": workspace_id, "workspace_name": name}
        if workspace_id in workspaces and workspaces[workspace_id] != workspace:
            raise ValueError("Conflicting workspace identity")
        workspaces[workspace_id] = workspace
    return list(workspaces.values())


def discover_collection_scope(fabric, admin, workspace_name=None, report_workspace_name=None, temporary_access=False):
    def select(workspaces, name):
        return [workspace for workspace in workspaces
                if name is None or workspace["workspace_name"].casefold() == name.casefold()]

    report_error = None
    if temporary_access:
        discovered = normalize_workspaces(
            admin.list_workspaces(workspace_state="Active", workspace_type="Workspace"), active_only=True
        )
        model_workspaces = select(discovered, workspace_name)
        report_workspaces = select(discovered, report_workspace_name)
        if not report_workspaces:
            raise ValueError("No report workspaces match the selected admin scope")
        report_scope = "admin_workspace_filter:" + report_workspace_name if report_workspace_name is not None else "admin_workspaces"
    else:
        try:
            discovered = normalize_workspaces(admin.list_workspaces())
        except (AttributeError, FabricHTTPException):
            discovered = normalize_workspaces(fabric.list_workspaces())
        model_workspaces = select(discovered, workspace_name)
        report_scope = "workspace_filter:" + report_workspace_name if report_workspace_name is not None else "visible_workspaces"
        try:
            report_workspaces = select(normalize_workspaces(fabric.list_workspaces()), report_workspace_name)
            if not report_workspaces:
                raise ValueError("No report workspaces in scope")
        except Exception as error:
            report_workspaces = []
            report_error = type(error).__name__
    if not model_workspaces:
        raise ValueError(f"No model workspaces match WORKSPACE_NAME={workspace_name!r}")

    selected = {}
    for flag, workspaces in (("scan_models", model_workspaces), ("scan_reports", report_workspaces)):
        for workspace in workspaces:
            target = selected.setdefault(workspace["workspace_id"], {
                **workspace, "scan_models": False, "scan_reports": False,
            })
            target[flag] = True
    return {"workspaces": list(selected.values()), "report_scope": report_scope, "report_error": report_error}


def new_catalog_rows():
    return {key: [] for key in ("models", "datasources", "tables", "columns", "relationships", "measures", "errors", "security", "queries")}


M_SOURCE_NAMESPACES = {
    "activedirectory", "amazonredshift", "analysisservices", "azuredataexplorer", "azuredatalakestorage",
    "azurestorage", "cdm", "commondataservice", "databricks", "dataverse", "db2", "deltalake", "essbase",
    "exchange", "fabric", "file", "folder", "googlebigquery", "hdfs", "hdinsight", "impala", "informix",
    "kusto", "lakehouse", "mysql", "odata", "odbc", "oledb", "oracle", "postgresql", "powerbi",
    "powerplatform", "salesforce", "saphana", "sharepoint", "snowflake", "spark", "sql", "sybase",
    "teradata", "vertica", "warehouse", "web",
}


def norm_m(value):
    # Power Query M normalized for comparison: comments removed, whitespace collapsed and case
    # folded outside string literals. Literals (server, database, path names) are kept verbatim.
    import re

    if value is None or (isinstance(value, float) and value != value):
        return ""
    pieces = []
    for match in re.finditer(r'"(?:[^"]|"")*"|/\*.*?\*/|//[^\n]*|[^"/]+|[/"]', str(value), flags=re.S):
        token = match.group(0)
        if token.startswith('"') and len(token) > 1:
            pieces.append(token)
        elif token.startswith("/*") or token.startswith("//"):
            pieces.append(" ")
        else:
            pieces.append(token.casefold())
    return re.sub(r"\s+", " ", "".join(pieces)).strip()


def m_parameter_value(expression):
    # Literal value of a Power Query parameter such as `"server" meta [IsParameterQuery=true]`.
    import re

    match = re.match(r'\s*"((?:[^"]|"")*)"\s*meta\b', str(expression or ""))
    return match.group(1).replace('""', '"') if match else None


def extract_m_sources(expression, parameters=None):
    # Upstream sources referenced by connector calls such as Sql.Database("server", "db").
    # Arguments that are parameter names resolve through `parameters`; unknown names become "?".
    import re

    literal = r'"(?:[^"]|"")*"'
    argument = rf'(?:{literal}|#{literal}|[a-z_][a-z0-9_.]*)'
    call = re.compile(rf'([a-z][a-z0-9]*(?:\.[a-z][a-z0-9]*)+)\s*\(\s*({argument}(?:\s*,\s*{argument})*)\s*[,)]')
    lookup = {str(key).casefold(): value for key, value in (parameters or {}).items() if value is not None}
    sources = set()
    for match in call.finditer(norm_m(expression)):
        name = match.group(1)
        if name.split(".")[0] not in M_SOURCE_NAMESPACES:
            continue
        resolved = []
        for arg in re.findall(argument, match.group(2)):
            if arg.startswith('"'):
                resolved.append(arg[1:-1].replace('""', '"').casefold())
            else:
                key = arg[2:-1].replace('""', '"') if arg.startswith("#") else arg
                value = lookup.get(key.casefold())
                resolved.append(str(value).casefold() if value is not None else "?")
        if any(value != "?" for value in resolved):
            sources.add(f"{name}({', '.join(resolved)})")
    return sources


def collect_workspace_models(fabric, connect_model, workspace, scan_id, scanned_at, model_filter=None, read=None):
    from contextlib import ExitStack

    rows = new_catalog_rows()
    read = read or (lambda operation: operation())
    workspace_id = workspace["workspace_id"]
    workspace_name = workspace["workspace_name"]
    try:
        datasets = read(lambda: fabric.list_datasets(workspace=workspace_id)).to_dict("records")
    except Exception as error:
        rows["errors"].append({
            "workspace_id": workspace_id, "workspace_name": workspace_name,
            "model_id": None, "model_name": None,
            "error_type": type(error).__name__, "error_message": f"Could not list semantic models: {error}",
        })
        return rows

    for dataset in datasets:
        model_id = str(dataset["Dataset ID"] if "Dataset ID" in dataset else dataset["Id"])
        model_name = str(dataset["Dataset Name"] if "Dataset Name" in dataset else dataset["Name"])
        if model_filter is not None and model_name.casefold() != model_filter.casefold():
            continue
        print(f"  Extracting model: {model_name}")
        identity = {"workspace_id": workspace_id, "workspace_name": workspace_name,
                    "model_id": model_id, "model_name": model_name}
        security_result = {
            "security_schema_version": 1, "scan_status": "failed", "error_type": None,
            "role_count": None, "rls_filter_count": None, "table_ols_count": None,
            "column_ols_count": None, "definition_json": None,
        }
        security_read = False
        try:
            with ExitStack() as connections:
                tom = read(lambda: connections.enter_context(connect_model(
                    dataset=model_id, workspace=workspace_id, readonly=True,
                )))
                model = tom.model
                security_result = collect_security_definition(model)
                security_read = True
                rows["models"].append({
                    **identity, "catalog_scan_id": scan_id,
                    "compatibility_level": getattr(model, "CompatibilityLevel", None),
                    "default_mode": str(getattr(model, "DefaultMode", None)) if getattr(model, "DefaultMode", None) is not None else None,
                })
                for datasource in model.DataSources:
                    rows["datasources"].append({
                        **identity,
                        "datasource_name": str(getattr(datasource, "Name", "") or ""),
                        "datasource_type": str(getattr(datasource, "Type", None)) if getattr(datasource, "Type", None) is not None else None,
                        "connection_string": getattr(datasource, "ConnectionString", None),
                        "connection_details": str(getattr(datasource, "ConnectionDetails", None)) if getattr(datasource, "ConnectionDetails", None) is not None else None,
                        "impersonation_mode": str(getattr(datasource, "ImpersonationMode", None)) if getattr(datasource, "ImpersonationMode", None) is not None else None,
                        "description": getattr(datasource, "Description", None),
                    })
                for expression in getattr(model, "Expressions", []):
                    rows["datasources"].append({
                        **identity, "datasource_name": str(getattr(expression, "Name", "") or ""),
                        "datasource_type": "M expression", "connection_string": getattr(expression, "Expression", None),
                        "connection_details": None, "impersonation_mode": None,
                        "description": getattr(expression, "Description", None),
                    })
                    expression_text = getattr(expression, "Expression", None)
                    expression_kind = getattr(expression, "Kind", None)
                    rows["queries"].append({
                        **identity, "table_name": None, "partition_name": str(getattr(expression, "Name", "") or ""),
                        "query_kind": "shared_expression",
                        "source_type": str(expression_kind) if expression_kind is not None else "M",
                        "expression": str(expression_text) if expression_text else None,
                    })
                for table in model.Tables:
                    for partition in getattr(table, "Partitions", []):
                        partition_source = getattr(partition, "Source", None)
                        m_expression = getattr(partition_source, "Expression", None) if partition_source is not None else None
                        source_type = type(partition_source).__name__ if partition_source is not None else None
                        # Non-M partitions (Direct Lake entity, calculated) are recorded without M text.
                        rows["queries"].append({
                            **identity, "table_name": str(getattr(table, "Name", "") or ""),
                            "partition_name": str(getattr(partition, "Name", "") or ""), "query_kind": "partition",
                            "source_type": source_type,
                            "expression": str(m_expression) if source_type == "MPartitionSource" and m_expression else None,
                        })
                        if not m_expression:
                            continue
                        rows["datasources"].append({
                            **identity, "datasource_name": str(getattr(partition, "Name", "") or ""),
                            "datasource_type": type(partition_source).__name__, "connection_string": str(m_expression),
                            "connection_details": None, "impersonation_mode": None, "description": None,
                        })
                for table in model.Tables:
                    table_name = str(getattr(table, "Name", "") or "")
                    rows["tables"].append({
                        **identity, "table_name": table_name, "description": getattr(table, "Description", None),
                        "is_hidden": getattr(table, "IsHidden", None),
                    })
                    for column in getattr(table, "Columns", []):
                        rows["columns"].append({
                            **identity, "table_name": table_name, "column_name": str(getattr(column, "Name", "") or ""),
                            "data_type": str(getattr(column, "DataType", None)) if getattr(column, "DataType", None) is not None else None,
                            "description": getattr(column, "Description", None), "is_hidden": getattr(column, "IsHidden", None),
                        })
                    for measure in getattr(table, "Measures", []):
                        rows["measures"].append({
                            **identity, "table_name": table_name, "measure_name": str(getattr(measure, "Name", "") or ""),
                            "expression": getattr(measure, "Expression", None), "format_string": getattr(measure, "FormatString", None),
                            "description": getattr(measure, "Description", None), "is_hidden": getattr(measure, "IsHidden", None),
                        })
                for relationship in getattr(model, "Relationships", []):
                    from_table = getattr(relationship, "FromTable", None)
                    from_column = getattr(relationship, "FromColumn", None)
                    to_table = getattr(relationship, "ToTable", None)
                    to_column = getattr(relationship, "ToColumn", None)
                    rows["relationships"].append({
                        **identity, "relationship_name": str(getattr(relationship, "Name", "") or ""),
                        "from_table": str(getattr(from_table, "Name", "") or ""),
                        "from_column": str(getattr(from_column, "Name", "") or ""),
                        "to_table": str(getattr(to_table, "Name", "") or ""),
                        "to_column": str(getattr(to_column, "Name", "") or ""),
                        "cross_filtering_behavior": str(getattr(relationship, "CrossFilteringBehavior", None)) if getattr(relationship, "CrossFilteringBehavior", None) is not None else None,
                        "is_active": getattr(relationship, "IsActive", None),
                    })
        except Exception as error:
            if not security_read:
                security_result["error_type"] = type(error).__name__
            rows["errors"].append({**identity, "error_type": type(error).__name__, "error_message": str(error)})
        finally:
            rows["security"].append({**identity, "catalog_scan_id": scan_id, "scanned_at": scanned_at, **security_result})
    return rows


In [ ]:
def collect_report_dependencies(client, workspaces, models, scan_id, scanned_at):
    from urllib.parse import quote, urljoin, urlsplit

    model_index = {str(model["model_id"]).strip().casefold(): model for model in models}
    dependencies = []
    scans = []
    for workspace in workspaces:
        workspace_id = str(workspace["workspace_id"]).strip().casefold()
        workspace_name = str(workspace["workspace_name"])
        endpoint = "/v1.0/myorg/groups/" + quote(workspace_id, safe="") + "/reports"
        next_url = "https://api.powerbi.com" + endpoint
        visited = set()
        workspace_reports = {}
        scan_status = "complete"
        error_type = None
        try:
            while next_url:
                parsed = urlsplit(next_url)
                if (parsed.scheme != "https" or parsed.netloc != "api.powerbi.com"
                        or parsed.path != endpoint or parsed.fragment or next_url in visited):
                    raise ValueError("Invalid report continuation")
                visited.add(next_url)
                response = client.get(parsed.path + ("?" + parsed.query if parsed.query else ""))
                response.raise_for_status()
                payload = response.json()
                if not isinstance(payload, dict) or not isinstance(payload.get("value"), list):
                    raise ValueError("Invalid report collection")
                for report in payload["value"]:
                    if not isinstance(report, dict) or not isinstance(report.get("id"), str) or not report["id"].strip():
                        raise ValueError("Missing report identity")
                    report_id = report["id"].strip().casefold()
                    raw_model_id = report.get("datasetId")
                    if raw_model_id is not None and not isinstance(raw_model_id, str):
                        raise ValueError("Invalid model identity")
                    model_id = raw_model_id.strip().casefold() if raw_model_id else None
                    report_type = report.get("reportType") or "PowerBIReport"
                    target = model_index.get(model_id)
                    if report_type != "PowerBIReport":
                        binding_status = "unsupported_report_type"
                    elif not model_id:
                        binding_status = "missing_dataset_id"
                    elif target is None:
                        binding_status = "uncataloged_model"
                    else:
                        binding_status = "cataloged_model"
                    dependency = {
                        "scan_id": scan_id,
                        "scanned_at": scanned_at,
                        "report_workspace_id": workspace_id,
                        "report_workspace_name": workspace_name,
                        "report_id": report_id,
                        "report_name": str(report.get("name") or report_id),
                        "report_type": str(report_type),
                        "report_url": "https://app.powerbi.com/groups/" + quote(workspace_id, safe="") + "/reports/" + quote(report_id, safe=""),
                        "model_id": model_id,
                        "model_workspace_id": str(target["workspace_id"]) if target else None,
                        "model_workspace_name": str(target["workspace_name"]) if target else None,
                        "model_name": str(target["model_name"]) if target else None,
                        "binding_status": binding_status,
                        "is_cross_workspace": workspace_id != str(target["workspace_id"]).casefold() if target else None,
                    }
                    if report_id in workspace_reports and workspace_reports[report_id]["model_id"] != model_id:
                        raise ValueError("Report binding changed during scan")
                    workspace_reports[report_id] = dependency
                continuation = payload.get("@odata.nextLink")
                if continuation is not None and not isinstance(continuation, str):
                    raise ValueError("Invalid report continuation")
                next_url = urljoin("https://api.powerbi.com" + endpoint, continuation) if continuation else None
        except Exception as error:
            scan_status = "partial" if workspace_reports else "failed"
            error_type = type(error).__name__
        records = list(workspace_reports.values())
        unresolved_count = sum(record["binding_status"] == "missing_dataset_id" for record in records)
        unsupported_count = sum(record["binding_status"] == "unsupported_report_type" for record in records)
        if scan_status == "complete" and (unresolved_count or unsupported_count):
            scan_status = "partial"
        dependencies.extend(records)
        scans.append({
            "scan_id": scan_id,
            "scanned_at": scanned_at,
            "report_workspace_id": workspace_id,
            "report_workspace_name": workspace_name,
            "scan_status": scan_status,
            "report_count": len(records),
            "bound_report_count": sum(record["binding_status"] in ("cataloged_model", "uncataloged_model") for record in records),
            "unresolved_report_count": unresolved_count,
            "unsupported_report_count": unsupported_count,
            "error_type": error_type,
        })
    return dependencies, scans

In [ ]:
def resolve_report_bindings(reports, models):
    model_index = {str(model["model_id"]).strip().casefold(): model for model in models}
    resolved = []
    for report in reports:
        target = model_index.get(report["model_id"])
        binding_status = report["binding_status"]
        if binding_status in {"cataloged_model", "uncataloged_model"}:
            binding_status = "cataloged_model" if target else "uncataloged_model"
        resolved.append({
            **report,
            "model_workspace_id": str(target["workspace_id"]) if target else None,
            "model_workspace_name": str(target["workspace_name"]) if target else None,
            "model_name": str(target["model_name"]) if target else None,
            "binding_status": binding_status,
            "is_cross_workspace": report["report_workspace_id"] != str(target["workspace_id"]).casefold() if target else None,
        })
    return resolved


def failed_report_scan(scan_id, scanned_at, scope, error_type, workspace=None):
    workspace = workspace or {}
    return {
        "scan_id": scan_id, "scanned_at": scanned_at,
        "report_workspace_id": workspace.get("workspace_id"),
        "report_workspace_name": workspace.get("workspace_name"),
        "scan_status": "failed", "report_count": 0, "bound_report_count": 0,
        "unresolved_report_count": 0, "unsupported_report_count": 0,
        "error_type": error_type, "scan_scope": scope,
    }


def collect_catalog(fabric, admin, connect_model, workspace_name=None, model_filter=None,
                    report_workspace_name=None, access=None, scan_id=None, scanned_at=None):
    from contextlib import nullcontext
    from datetime import datetime, timezone
    from types import SimpleNamespace
    from uuid import uuid4

    scan_id = scan_id or str(uuid4())
    scanned_at = scanned_at or datetime.now(timezone.utc).isoformat()
    scope = discover_collection_scope(fabric, admin, workspace_name, report_workspace_name, access is not None)
    rows = new_catalog_rows()
    reports, scans = [], []
    if scope["report_error"]:
        scans.append(failed_report_scan(scan_id, scanned_at, scope["report_scope"], scope["report_error"]))
    print(f"Collection scope: {len(scope['workspaces'])} workspaces; reports: {scope['report_scope']}")
    if access is not None:
        access.begin(scan_id, scope)

    def record_error(workspace, error):
        rows["errors"].append({
            "workspace_id": workspace["workspace_id"], "workspace_name": workspace["workspace_name"],
            "model_id": None, "model_name": None, "error_type": type(error).__name__, "error_message": str(error),
        })

    def report_client(read):
        client = fabric.PowerBIRestClient()
        if read is None:
            return client

        def get(url):
            def request():
                response = client.get(url)
                response.raise_for_status()
                return response
            return read(request)
        return SimpleNamespace(get=get)

    for workspace in scope["workspaces"]:
        print(f"Scanning workspace: {workspace['workspace_name']}")
        try:
            with access.workspace(workspace) if access is not None else nullcontext(None) as read:
                if workspace["scan_models"]:
                    try:
                        collected = collect_workspace_models(fabric, connect_model, workspace, scan_id, scanned_at, model_filter, read)
                        for key in rows:
                            rows[key].extend(collected[key])
                    except Exception as error:
                        record_error(workspace, error)
                if workspace["scan_reports"]:
                    workspace_reports, workspace_scans = collect_report_dependencies(
                        report_client(read), [workspace], [], scan_id, scanned_at,
                    )
                    reports.extend(workspace_reports)
                    for scan in workspace_scans:
                        scan["scan_scope"] = scope["report_scope"]
                    scans.extend(workspace_scans)
        except WorkspaceAccessError:
            raise
        except Exception as error:
            record_error(workspace, error)
            if workspace["scan_reports"]:
                scans.append(failed_report_scan(scan_id, scanned_at, scope["report_scope"], type(error).__name__, workspace))
    if access is not None:
        access.finish()
    return {"rows": rows, "reports": resolve_report_bindings(reports, rows["models"]),
            "report_scans": scans, "scope": scope, "scan_id": scan_id, "scanned_at": scanned_at}


catalog_result = collect_catalog(
    fabric, admin, connect_semantic_model, WORKSPACE_NAME, MODEL_NAME, REPORT_WORKSPACE_NAME,
    access=workspace_access,
)
catalog_scan_id = catalog_result["scan_id"]
catalog_scanned_at = catalog_result["scanned_at"]
models_df = pd.DataFrame(catalog_result["rows"]["models"])
datasources_df = pd.DataFrame(catalog_result["rows"]["datasources"])
tables_df = pd.DataFrame(catalog_result["rows"]["tables"])
columns_df = pd.DataFrame(catalog_result["rows"]["columns"])
relationships_df = pd.DataFrame(catalog_result["rows"]["relationships"])
measures_df = pd.DataFrame(catalog_result["rows"]["measures"])
queries_df = pd.DataFrame(catalog_result["rows"]["queries"])
errors_df = pd.DataFrame(catalog_result["rows"]["errors"])
security_df = pd.DataFrame(catalog_result["rows"]["security"])
reports_df = pd.DataFrame(catalog_result["reports"])
report_scans_df = pd.DataFrame(catalog_result["report_scans"])
print(f"Cataloged {len(models_df)} models across {sum(workspace['scan_models'] for workspace in catalog_result['scope']['workspaces'])} model workspaces.")
print(f"Datasources: {len(datasources_df)} | Tables: {len(tables_df)} | Columns: {len(columns_df)} | Relationships: {len(relationships_df)} | Measures: {len(measures_df)} | M queries: {int(queries_df['expression'].notna().sum()) if not queries_df.empty else 0}")
print(f"Security definitions complete: {sum(row['scan_status'] == 'complete' for row in catalog_result['rows']['security'])}/{len(security_df)} model scans.")
print(f"Report snapshot: {len(reports_df)} reports; {sum(scan['scan_status'] == 'complete' for scan in catalog_result['report_scans'])}/{len(report_scans_df)} report workspace scans complete.")
if not errors_df.empty:
    print(f"Workspace/model errors: {len(errors_df)}")


## Review and Persist the Catalog

Collection has finished and any temporary workspace assignments have been removed and checked. Review workspace/model errors and security/report completeness, then persist the catalog. Ordinary scan failures are recorded; unresolved access cleanup stops execution before this stage.

The eleven catalog Delta tables use explicit schemas and overwrite the previous snapshot, including empty results. The scoring stage below reloads these registered tables from the same Lakehouse.

In [ ]:
# Per-workspace model counts give a quick overview before the detailed frames.
if not models_df.empty:
    workspace_summary = (
        models_df.groupby("workspace_name")
        .size()
        .reset_index(name="model_count")
        .sort_values("model_count", ascending=False)
        .reset_index(drop=True)
    )
    display(workspace_summary)

display(models_df)
display(datasources_df)
display(tables_df)
display(columns_df)
display(relationships_df)
display(measures_df)
display(queries_df)
display(errors_df)

In [ ]:
catalog_outputs = {
    "semantic_models": models_df,
    "semantic_model_datasources": datasources_df,
    "semantic_model_tables": tables_df,
    "semantic_model_columns": columns_df,
    "semantic_model_relationships": relationships_df,
    "semantic_model_measures": measures_df,
    "semantic_model_queries": queries_df,
    "semantic_model_catalog_errors": errors_df,
    "semantic_model_report_dependencies": reports_df,
    "semantic_model_report_scan": report_scans_df,
    "semantic_model_security": security_df,
}

identity_schema = "workspace_id STRING, workspace_name STRING, model_id STRING, model_name STRING"
catalog_schemas = {
    "semantic_models": identity_schema + ", compatibility_level BIGINT, default_mode STRING, catalog_scan_id STRING",
    "semantic_model_datasources": identity_schema + ", datasource_name STRING, datasource_type STRING, connection_string STRING, connection_details STRING, impersonation_mode STRING, description STRING",
    "semantic_model_tables": identity_schema + ", table_name STRING, description STRING, is_hidden BOOLEAN",
    "semantic_model_columns": identity_schema + ", table_name STRING, column_name STRING, data_type STRING, description STRING, is_hidden BOOLEAN",
    "semantic_model_relationships": identity_schema + ", relationship_name STRING, from_table STRING, from_column STRING, to_table STRING, to_column STRING, cross_filtering_behavior STRING, is_active BOOLEAN",
    "semantic_model_measures": identity_schema + ", table_name STRING, measure_name STRING, expression STRING, format_string STRING, description STRING, is_hidden BOOLEAN",
    "semantic_model_queries": identity_schema + ", table_name STRING, partition_name STRING, query_kind STRING, source_type STRING, expression STRING, expression_hash STRING",
    "semantic_model_catalog_errors": identity_schema + ", error_type STRING, error_message STRING",
    "semantic_model_report_dependencies": "scan_id STRING, scanned_at STRING, report_workspace_id STRING, report_workspace_name STRING, report_id STRING, report_name STRING, report_type STRING, report_url STRING, model_id STRING, model_workspace_id STRING, model_workspace_name STRING, model_name STRING, binding_status STRING, is_cross_workspace BOOLEAN",
    "semantic_model_report_scan": "scan_id STRING, scanned_at STRING, report_workspace_id STRING, report_workspace_name STRING, scan_status STRING, report_count BIGINT, bound_report_count BIGINT, unresolved_report_count BIGINT, unsupported_report_count BIGINT, error_type STRING, scan_scope STRING",
    "semantic_model_security": identity_schema + ", catalog_scan_id STRING, scanned_at STRING, security_schema_version BIGINT, scan_status STRING, error_type STRING, role_count BIGINT, rls_filter_count BIGINT, table_ols_count BIGINT, column_ols_count BIGINT, definition_json STRING",
}

for table_name, frame in catalog_outputs.items():
    records = frame.to_dict("records")
    if table_name == "semantic_model_queries":
        import hashlib

        for record in records:
            normalized = norm_m(record.get("expression"))
            record["expression_hash"] = hashlib.sha256(normalized.encode("utf-8")).hexdigest() if normalized else None
    for record in records:
        for field, value in record.items():
            if pd.isna(value):
                record[field] = None
        for field in ("compatibility_level", "security_schema_version", "role_count", "rls_filter_count", "table_ols_count", "column_ols_count"):
            if record.get(field) is not None:
                record[field] = int(record[field])
    spark.createDataFrame(records, schema=catalog_schemas[table_name]).write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(table_name)
    print(f"Wrote {len(frame)} rows to {table_name}")


## Prepare the data

Loads the catalog and scores every model pair. All inputs here are hidden — expand any cell to inspect the logic; you don't need to change anything.

In [ ]:
import itertools
import re
from collections import defaultdict

import numpy as np
import pandas as pd

In [ ]:
import hashlib
import json
import math


def security_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True, allow_nan=False)


def normalize_score_weights(weights):
    if not weights or any(isinstance(value, bool) or not math.isfinite(value) or value < 0 for value in weights.values()):
        raise ValueError("Score weights must be finite nonnegative numbers.")
    total = math.fsum(weights.values())
    if total <= 0:
        raise ValueError("Score weights must have a positive sum.")
    return {key: value / total for key, value in weights.items()}


def build_security_signature(identity, snapshots, load_status="available"):
    result = {"status": load_status, "fingerprint": None, "roles": [], "propagation": [], "definition_json": None}
    if load_status != "available":
        return result
    if len(snapshots) != 1:
        result["status"] = "not_collected" if not snapshots else "inconsistent"
        return result
    snapshot = snapshots[0]
    if (not isinstance(identity.get("catalog_scan_id"), str) or not identity["catalog_scan_id"]
            or any(snapshot.get(key) != identity.get(key) for key in ("model_id", "workspace_id", "catalog_scan_id"))):
        result["status"] = "inconsistent"
        return result
    if snapshot.get("security_schema_version") != 1:
        result["status"] = "unsupported"
        return result
    if snapshot.get("scan_status") != "complete":
        result["status"] = "incomplete"
        return result

    def text(value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError("Missing security identity")
        return value

    def permission(value, allowed):
        if value not in allowed:
            raise ValueError("Invalid security permission")
        return value

    try:
        definition = json.loads(snapshot["definition_json"])
        if not isinstance(definition["roles"], list) or not isinstance(definition["relationships"], list):
            raise ValueError("Invalid security inventory")
        roles, role_names = [], set()
        filter_count = table_ols_count = column_ols_count = 0
        for role in definition["roles"]:
            role_name = text(role["name"])
            if role_name in role_names or not isinstance(role["tables"], list):
                raise ValueError("Invalid role inventory")
            role_names.add(role_name)
            model_permission = permission(role["model_permission"], {"None", "Read", "ReadRefresh", "Refresh", "Administrator"})
            rls, table_ols, column_ols, table_names = set(), set(), set(), set()
            for table in role["tables"]:
                table_name = text(table["table"])
                if table_name in table_names or not isinstance(table["columns"], list):
                    raise ValueError("Invalid table permission inventory")
                table_names.add(table_name)
                expression = table["filter_expression"]
                if not isinstance(expression, str):
                    raise ValueError("Invalid RLS definition")
                if expression.strip():
                    rls.add((table_name, expression))
                    filter_count += 1
                table_permission = permission(table["metadata_permission"], {"Default", "None", "Read"})
                if table_permission != "Default":
                    table_ols.add((table_name, table_permission))
                table_ols_count += table_permission == "None"
                column_names = set()
                for column in table["columns"]:
                    column_name = text(column["column"])
                    if column_name in column_names:
                        raise ValueError("Duplicate column permission")
                    column_names.add(column_name)
                    column_permission = permission(column["metadata_permission"], {"Default", "None", "Read"})
                    if column_permission != "Default":
                        column_ols.add((table_name, column_name, column_permission))
                    column_ols_count += column_permission == "None"
            bundle = {
                "model_permission": model_permission,
                "rls": sorted(rls), "table_ols": sorted(table_ols), "column_ols": sorted(column_ols),
            }
            role_key = security_json(bundle)
            roles.append({"name": role_name, "key": role_key, **bundle})
        roles.sort(key=lambda role: (role["key"], role["name"]))
        propagation = set()
        for relationship in definition["relationships"]:
            endpoints = tuple(text(relationship[key]) for key in ("from_table", "from_column", "to_table", "to_column"))
            if type(relationship["is_active"]) is not bool:
                raise ValueError("Invalid relationship active state")
            settings = (
                relationship["is_active"],
                permission(relationship["cross_filtering_behavior"], {"OneDirection", "BothDirections", "Automatic"}),
                permission(relationship["security_filtering_behavior"], {"OneDirection", "BothDirections", "None"}),
                permission(relationship["from_cardinality"], {"One", "Many"}),
                permission(relationship["to_cardinality"], {"One", "Many"}),
            )
            if filter_count:
                propagation.add(endpoints + settings)
        expected_counts = {"role_count": len(roles), "rls_filter_count": filter_count, "table_ols_count": table_ols_count, "column_ols_count": column_ols_count}
        if any(snapshot.get(key) != value for key, value in expected_counts.items()):
            raise ValueError("Security inventory counts do not agree")
        canonical = security_json({"version": 1, "roles": [role["key"] for role in roles], "propagation": sorted(propagation)})
        for role in roles:
            role["fingerprint"] = hashlib.sha256(role.pop("key").encode("utf-8")).hexdigest()
        result.update({
            "status": "complete", "fingerprint": hashlib.sha256(canonical.encode("utf-8")).hexdigest(),
            "roles": roles, "propagation": sorted(propagation), **expected_counts,
            "definition_json": security_json({"roles": roles, "propagation": sorted(propagation)}),
        })
    except (KeyError, TypeError, ValueError, AttributeError):
        result["status"] = "inconsistent"
    return result


def security_overlap(left, right):
    union = left | right
    return len(left & right) / len(union) if union else None


def security_weighted_average(components, weights):
    applicable = {key: weights[key] for key, value in components.items() if value is not None}
    effective = normalize_score_weights(applicable)
    return math.fsum(components[key] * weight for key, weight in effective.items()), effective


def score_security_pair(left, right, role_weights, security_weights):
    evidence = {"components": {}, "effective_weights": {}, "role_matches": [], "unmatched_roles_a": [], "unmatched_roles_b": []}
    if left["status"] != "complete" or right["status"] != "complete":
        return {"score": None, "status": "unknown", "evidence": evidence}
    roles_a, roles_b = left["roles"], right["roles"]
    if not roles_a and not roles_b:
        return {"score": None, "status": "not_applicable", "evidence": evidence}
    comparisons = []
    for role_a in roles_a:
        row = []
        for role_b in roles_b:
            components = {"model_permission": float(role_a["model_permission"] == role_b["model_permission"])}
            for family in ("rls", "table_ols", "column_ols"):
                components[family] = security_overlap({tuple(fact) for fact in role_a[family]}, {tuple(fact) for fact in role_b[family]})
            value, effective = security_weighted_average(components, role_weights)
            row.append({"score": value, "components": components, "effective_weights": effective})
        comparisons.append(row)
    matches = []
    if roles_a and roles_b:
        from scipy.optimize import linear_sum_assignment

        indices_a, indices_b = linear_sum_assignment([[-item["score"] for item in row] for row in comparisons])
        matches = [{"role_a": int(index_a), "role_b": int(index_b), **comparisons[index_a][index_b]}
                   for index_a, index_b in zip(indices_a, indices_b)]
    matched_a = {match["role_a"] for match in matches}
    matched_b = {match["role_b"] for match in matches}
    role_score = math.fsum(match["score"] for match in matches) / max(len(roles_a), len(roles_b))
    propagation_score = security_overlap({tuple(fact) for fact in left["propagation"]}, {tuple(fact) for fact in right["propagation"]})
    components = {"role_definitions": role_score, "rls_propagation": propagation_score}
    value, effective = security_weighted_average(components, security_weights)
    evidence.update({
        "components": components, "effective_weights": effective, "role_matches": matches,
        "unmatched_roles_a": [index for index in range(len(roles_a)) if index not in matched_a],
        "unmatched_roles_b": [index for index in range(len(roles_b)) if index not in matched_b],
    })
    return {"score": value, "status": "match" if left["fingerprint"] == right["fingerprint"] else "different", "evidence": evidence}


def combine_similarity(schema_score, security_result, weights):
    effective = normalize_score_weights(weights)
    if set(effective) != {"schema", "security"}:
        raise ValueError("Combined weights must specify schema and security.")
    result = {"combined_score": None, "score_mode": "unavailable_security", "effective_weights": {}}
    if not isinstance(schema_score, (int, float)) or not math.isfinite(schema_score) or not 0 <= schema_score <= 1:
        return result
    if security_result["status"] == "not_applicable":
        result.update(combined_score=schema_score, score_mode="schema_only_no_security", effective_weights={"schema": 1.0, "security": 0.0})
    elif security_result["status"] in ("match", "different"):
        security_score = security_result["score"]
        if isinstance(security_score, (int, float)) and math.isfinite(security_score) and 0 <= security_score <= 1:
            combined = math.fsum((effective["schema"] * schema_score, effective["security"] * security_score))
            result.update(combined_score=combined, score_mode="combined", effective_weights=effective)
    return result


def classify_similarity(value, duplicate_threshold, similar_threshold):
    if value is None or not math.isfinite(value) or not 0 <= value <= 1:
        return "unassessed"
    if value >= duplicate_threshold:
        return "duplicate"
    if value >= similar_threshold:
        return "similar"
    return "distinct"


In [ ]:
def load_delta(table_name):
    return spark.table(table_name).toPandas()


models_df = load_delta("semantic_models")
tables_df = load_delta("semantic_model_tables")
columns_df = load_delta("semantic_model_columns")
relationships_df = load_delta("semantic_model_relationships")
measures_df = load_delta("semantic_model_measures")
datasources_df = load_delta("semantic_model_datasources")
queries_df = load_delta("semantic_model_queries")

if models_df.empty:
    raise ValueError(
        "The current catalog contains no readable semantic models. "
        "Review filters and semantic_model_catalog_errors, then rerun this notebook. "
        "No new similarity results were produced."
    )

print(f"Models: {len(models_df)}")
print(
    f"Tables: {len(tables_df)} | Columns: {len(columns_df)} | "
    f"Measures: {len(measures_df)} | Relationships: {len(relationships_df)} | "
    f"Datasources: {len(datasources_df)} | M queries: {int(queries_df['expression'].notna().sum()) if not queries_df.empty else 0}"
)


In [ ]:
def norm(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().casefold()


def norm_dax(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value)
    text = re.sub(r"/\*.*?\*/", " ", text, flags=re.S)  # block comments
    text = re.sub(r"//.*", " ", text)  # line comments
    text = re.sub(r"\s+", " ", text)
    return text.strip().casefold()


def model_label(sig):
    return f"{sig['workspace_name']} / {sig['model_name']}"


signatures = {}
for _, row in models_df.iterrows():
    model_id = str(row["model_id"])
    signatures[model_id] = {
        "model_id": model_id,
        "workspace_id": str(row.get("workspace_id", "")),
        "workspace_name": str(row.get("workspace_name", "")),
        "model_name": str(row.get("model_name", "")),
        "tables": set(),
        "columns": set(),
        "measure_names": set(),
        "measure_definitions": set(),
        "relationships": set(),
        "datasources": set(),
        "power_queries": set(),
        "m_docs": [],
        "dax_docs": [],
    }


def sig_for(model_id):
    return signatures.get(str(model_id))


for _, row in tables_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        sig["tables"].add(norm(row["table_name"]))

for _, row in columns_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        sig["columns"].add(f"{norm(row['table_name'])}.{norm(row['column_name'])}")

for _, row in measures_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        measure_name = norm(row["measure_name"])
        measure_dax = norm_dax(row.get("expression"))
        sig["measure_names"].add(measure_name)
        # Name + DAX key so containment only credits measures whose logic also matches.
        sig["measure_definitions"].add(f"{measure_name} :: {measure_dax}")
        sig["dax_docs"].append(f"{measure_name} {measure_dax}".strip())

for _, row in relationships_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        key = (
            f"{norm(row['from_table'])}.{norm(row['from_column'])}"
            f"->{norm(row['to_table'])}.{norm(row['to_column'])}"
        )
        sig["relationships"].add(key)

# Source definitions exclude M text, which is scored separately as Power Query; M contributes the
# upstream sources its connector calls reference instead.
M_DATASOURCE_TYPES = {"M expression", "MPartitionSource"}
for _, row in datasources_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None and row.get("datasource_type") not in M_DATASOURCE_TYPES:
        conn = row.get("connection_string") or row.get("connection_details") or row.get("datasource_name")
        conn_norm = norm(conn)
        if conn_norm:
            sig["datasources"].add(conn_norm)

query_records = queries_df.to_dict("records") if not queries_df.empty else []
m_parameters = defaultdict(dict)
for row in query_records:
    if row.get("query_kind") == "shared_expression" and row.get("expression"):
        value = m_parameter_value(row["expression"])
        if value is not None:
            m_parameters[str(row["model_id"])][str(row.get("partition_name") or "")] = value

for row in query_records:
    sig = sig_for(row["model_id"])
    normalized = norm_m(row.get("expression"))
    if sig is None or not normalized:
        continue
    if row.get("query_kind") == "shared_expression":
        sig["power_queries"].add(f"expression {norm(row.get('partition_name'))} :: {normalized}")
    else:
        sig["power_queries"].add(f"{norm(row.get('table_name'))} :: {normalized}")
    sig["m_docs"].append(normalized)
    sig["datasources"].update(extract_m_sources(row["expression"], m_parameters[str(row["model_id"])]))

# Build the embedding document per model, with a structural fallback when no measures exist.
for sig in signatures.values():
    parts = list(sig["dax_docs"])
    if not parts:
        parts = sorted(sig["tables"]) + sorted(sig["columns"])
    sig["doc"] = " \n ".join(parts) if parts else (sig["model_name"] or sig["model_id"])
    sig["m_doc"] = " \n ".join(sorted(sig["m_docs"]))

model_ids = list(signatures.keys())
print(f"Built signatures for {len(model_ids)} models.")


In [ ]:
def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 0.0
    union = len(set_a | set_b)
    return len(set_a & set_b) / union if union else 0.0


def weighted_schema_score(signal_scores, weights):
    # Weighted mean over applicable signals; a None score (not applicable) drops its weight.
    applicable = {signal: weight for signal, weight in weights.items() if signal_scores.get(signal) is not None}
    total = sum(applicable.values())
    return sum(weight * signal_scores[signal] for signal, weight in applicable.items()) / total if total else 0.0


def power_query_similarity(m_doc_a, m_doc_b, vector_a=None, vector_b=None):
    # Power Query text similarity: not applicable when neither model has M, 0 when only one has M.
    if not m_doc_a and not m_doc_b:
        return None, "not_applicable"
    if not m_doc_a or not m_doc_b:
        return 0.0, "one_sided"
    if vector_a is None or vector_b is None:
        return (1.0 if m_doc_a == m_doc_b else 0.0), "compared"
    return max(0.0, min(1.0, float(sum(a * b for a, b in zip(vector_a, vector_b))))), "compared"


def coverage(source_set, other_set):
    # Directional: the fraction of source_set's members that also appear in other_set.
    # Returns None when the signal is absent from the source, so it can be excluded from
    # the weighted average rather than counted as a spurious perfect match.
    if not source_set:
        return None
    return len(source_set & other_set) / len(source_set)


def weighted_containment(source_sig, other_sig, weights):
    # How completely source_sig is contained in other_sig: a weighted mean of the
    # per-signal coverages, normalized over whichever signals the source actually has.
    accumulated = 0.0
    total_weight = 0.0
    for signal, weight in weights.items():
        signal_coverage = coverage(source_sig[signal], other_sig[signal])
        if signal_coverage is None:
            continue
        accumulated += weight * signal_coverage
        total_weight += weight
    return accumulated / total_weight if total_weight else 0.0


def classify_containment(a_in_b, b_in_a, threshold):
    a_contained = a_in_b >= threshold
    b_contained = b_in_a >= threshold
    if a_contained and b_contained:
        return "equivalent"
    if b_contained:
        return "model_a_contains_model_b"
    if a_contained:
        return "model_b_contains_model_a"
    return "partial_overlap"


if ENABLE_BLOCKING:
    block_index = defaultdict(set)
    for model_id, sig in signatures.items():
        for table_name in sig["tables"]:
            block_index[("t", table_name)].add(model_id)
        for measure_name in sig["measure_names"]:
            block_index[("m", measure_name)].add(model_id)
    candidate_pairs = set()
    for group in block_index.values():
        if len(group) > 1:
            candidate_pairs.update(itertools.combinations(sorted(group), 2))
else:
    candidate_pairs = set(itertools.combinations(sorted(model_ids), 2))

print(f"Candidate pairs to score: {len(candidate_pairs)}")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF avoids the torch/transformers dependency chain (the Fabric runtime ships
# an older PyTorch than recent transformers require). Rows are L2-normalized, so
# cosine similarity stays a plain dot product for the downstream scoring step.
docs = [signatures[model_id]["doc"] for model_id in model_ids]
vectorizer = TfidfVectorizer(min_df=1, norm="l2")

embeddings = vectorizer.fit_transform(docs).toarray()
print(f"Encoded {len(docs)} model documents into {embeddings.shape[1]}-dim TF-IDF vectors.")
embedding_index = {model_id: idx for idx, model_id in enumerate(model_ids)}

# Power Query uses its own vocabulary so M tokens do not dilute DAX similarity.
m_model_ids = [model_id for model_id in model_ids if signatures[model_id]["m_doc"]]
m_embeddings = {}
if m_model_ids:
    try:
        m_vectorizer = TfidfVectorizer(min_df=1, norm="l2", token_pattern=r"(?u)\b\w+\b")
        m_matrix = m_vectorizer.fit_transform([signatures[model_id]["m_doc"] for model_id in m_model_ids]).toarray()
        m_embeddings = {model_id: m_matrix[idx] for idx, model_id in enumerate(m_model_ids)}
    except ValueError:
        m_embeddings = {}
print(f"Encoded Power Query for {len(m_embeddings)} of {len(model_ids)} models.")

In [ ]:
from uuid import uuid4
from pyspark.sql.utils import AnalysisException

analysis_run_id = str(uuid4())
security_rows, security_load_status = [], "available"
try:
    security_rows = load_delta("semantic_model_security").to_dict("records")
except AnalysisException as error:
    error_class = getattr(error, "getErrorClass", lambda: "")()
    security_load_status = "not_collected" if error_class in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND", "DELTA_PATH_DOES_NOT_EXIST") else "unavailable"

security_by_model = defaultdict(list)
identities_by_model = defaultdict(list)
for row in security_rows:
    security_by_model[str(row.get("model_id"))].append(row)
for row in models_df.to_dict("records"):
    identities_by_model[str(row["model_id"])].append(row)
for model_id, sig in signatures.items():
    identities = identities_by_model[model_id]
    identity = identities[0]
    scan_id = identity.get("catalog_scan_id")
    sig["catalog_scan_id"] = scan_id if len(identities) == 1 and isinstance(scan_id, str) and scan_id else None
    sig["security"] = build_security_signature(
        identity, security_by_model[model_id], security_load_status if len(identities) == 1 else "inconsistent"
    )
catalog_scan_ids = {sig["catalog_scan_id"] for sig in signatures.values()}
source_catalog_scan_id = next(iter(catalog_scan_ids)) if len(catalog_scan_ids) == 1 else None

pair_rows = []

for model_id_a, model_id_b in sorted(candidate_pairs):
    sig_a = signatures[model_id_a]
    sig_b = signatures[model_id_b]

    j_tables = jaccard(sig_a["tables"], sig_b["tables"])
    j_columns = jaccard(sig_a["columns"], sig_b["columns"])
    j_measures = jaccard(sig_a["measure_names"], sig_b["measure_names"])
    j_relationships = jaccard(sig_a["relationships"], sig_b["relationships"])
    j_datasources = jaccard(sig_a["datasources"], sig_b["datasources"])
    cosine = float(
        np.dot(embeddings[embedding_index[model_id_a]], embeddings[embedding_index[model_id_b]])
    )
    cosine = max(0.0, min(1.0, cosine))
    power_query, power_query_status = power_query_similarity(
        sig_a["m_doc"], sig_b["m_doc"], m_embeddings.get(model_id_a), m_embeddings.get(model_id_b)
    )

    composite = weighted_schema_score({
        "tables": j_tables,
        "columns": j_columns,
        "measure_names": j_measures,
        "measure_dax_embedding": cosine,
        "relationships": j_relationships,
        "datasources": j_datasources,
        "power_query": power_query,
    }, SIMILARITY_WEIGHTS)

    # Directional containment: how much of each model is absorbed by the other.
    a_in_b = weighted_containment(sig_a, sig_b, CONTAINMENT_WEIGHTS)
    b_in_a = weighted_containment(sig_b, sig_a, CONTAINMENT_WEIGHTS)
    containment_score = max(a_in_b, b_in_a)
    containment_relationship = classify_containment(a_in_b, b_in_a, CONTAINMENT_THRESHOLD)

    security_result = score_security_pair(sig_a["security"], sig_b["security"], SECURITY_ROLE_WEIGHTS, SECURITY_WEIGHTS)
    combination = combine_similarity(float(composite), security_result, COMBINED_WEIGHTS)
    combined = combination["combined_score"]
    tier = classify_similarity(combined, DUPLICATE_THRESHOLD, SIMILAR_THRESHOLD)
    security_evidence = {**security_result["evidence"], "combined_effective_weights": combination["effective_weights"]}

    pair_rows.append({
        "model_id_a": model_id_a,
        "model_a": model_label(sig_a),
        "workspace_a": sig_a["workspace_name"],
        "model_id_b": model_id_b,
        "model_b": model_label(sig_b),
        "workspace_b": sig_b["workspace_name"],
        "same_model_name": norm(sig_a["model_name"]) == norm(sig_b["model_name"]),
        "cross_workspace": sig_a["workspace_id"] != sig_b["workspace_id"],
        "jaccard_tables": round(j_tables, 4),
        "jaccard_columns": round(j_columns, 4),
        "jaccard_measure_names": round(j_measures, 4),
        "jaccard_relationships": round(j_relationships, 4),
        "jaccard_datasources": round(j_datasources, 4),
        "dax_embedding_cosine": round(cosine, 4),
        "composite_score": round(composite, 4),
        "schema_score": float(composite),
        "security_score": security_result["score"],
        "combined_score": combined,
        "score_mode": combination["score_mode"],
        "security_comparison_status": security_result["status"],
        "security_evidence_json": security_json(security_evidence),
        "security_fingerprint_a": sig_a["security"]["fingerprint"],
        "security_fingerprint_b": sig_b["security"]["fingerprint"],
        "catalog_scan_id_a": sig_a["catalog_scan_id"],
        "catalog_scan_id_b": sig_b["catalog_scan_id"],
        "analysis_run_id": analysis_run_id,
        "score_version": SCORE_VERSION,
        "containment_score": round(containment_score, 4),
        "containment_relationship": containment_relationship,
        "model_a_in_model_b": round(a_in_b, 4),
        "model_b_in_model_a": round(b_in_a, 4),
        "tier": tier,
        "power_query_similarity": round(power_query, 4) if power_query is not None else None,
        "power_query_status": power_query_status,
        "shared_query_count": len(sig_a["power_queries"] & sig_b["power_queries"]),
    })

pairs_df = pd.DataFrame(pair_rows)
if not pairs_df.empty:
    pairs_df = pairs_df.sort_values(["combined_score", "schema_score"], ascending=[False, False], na_position="last").reset_index(drop=True)

# Union-find clustering over duplicate-tier pairs.
parent = {model_id: model_id for model_id in model_ids}


def find(node):
    while parent[node] != node:
        parent[node] = parent[parent[node]]
        node = parent[node]
    return node


def union(node_a, node_b):
    root_a, root_b = find(node_a), find(node_b)
    if root_a != root_b:
        parent[root_a] = root_b


if not pairs_df.empty:
    for _, row in pairs_df[pairs_df["tier"] == "duplicate"].iterrows():
        union(row["model_id_a"], row["model_id_b"])

cluster_members = defaultdict(list)
for model_id in model_ids:
    cluster_members[find(model_id)].append(model_id)

cluster_rows = []
cluster_number = 0
for members in cluster_members.values():
    if len(members) > 1:
        cluster_number += 1
        for model_id in members:
            sig = signatures[model_id]
            cluster_rows.append({
                "cluster_id": cluster_number,
                "cluster_size": len(members),
                "model_id": model_id,
                "model": model_label(sig),
                "workspace_name": sig["workspace_name"],
                "model_name": sig["model_name"],
                "analysis_run_id": analysis_run_id,
                "score_version": SCORE_VERSION,
            })

clusters_df = pd.DataFrame(cluster_rows)

# Per-model signature summary.
signature_rows = []
for sig in signatures.values():
    security = sig["security"]
    signature_rows.append({
        "model_id": sig["model_id"],
        "workspace_id": sig["workspace_id"],
        "workspace_name": sig["workspace_name"],
        "model_name": sig["model_name"],
        "table_count": len(sig["tables"]),
        "column_count": len(sig["columns"]),
        "measure_count": len(sig["measure_names"]),
        "relationship_count": len(sig["relationships"]),
        "datasource_count": len(sig["datasources"]),
        "analysis_run_id": analysis_run_id,
        "catalog_scan_id": sig["catalog_scan_id"],
        "score_version": SCORE_VERSION,
        "security_schema_version": 1,
        "security_scan_status": security["status"],
        "security_fingerprint": security["fingerprint"],
        "security_definition_json": security["definition_json"],
        "role_count": security.get("role_count"),
        "rls_filter_count": security.get("rls_filter_count"),
        "table_ols_count": security.get("table_ols_count"),
        "column_ols_count": security.get("column_ols_count"),
        "query_count": len(sig["power_queries"]),
    })
signatures_df = pd.DataFrame(signature_rows)

duplicate_count = int((pairs_df["tier"] == "duplicate").sum()) if not pairs_df.empty else 0
similar_count = int((pairs_df["tier"] == "similar").sum()) if not pairs_df.empty else 0
unassessed_count = int((pairs_df["tier"] == "unassessed").sum()) if not pairs_df.empty else 0
containment_count = (
    int((pairs_df["containment_score"] >= CONTAINMENT_THRESHOLD).sum()) if not pairs_df.empty else 0
)
print(f"Possible duplicate pairs: {duplicate_count} | Similar pairs: {similar_count} | Schema coverage pairs: {containment_count}")
print(f"Unassessed combined scores: {unassessed_count} | Duplicate clusters: {cluster_number}")


## Save results to the lakehouse

Write the per-model signature summary, the full pairwise scores, and the duplicate clusters as Delta tables so the analysis is queryable outside this notebook.

In [ ]:
similarity_outputs = {
    "semantic_model_signatures": signatures_df,
    "semantic_model_similarity_pairs": pairs_df,
    "semantic_model_duplicate_clusters": clusters_df,
}
similarity_schemas = {
    "semantic_model_signatures": "model_id STRING, workspace_id STRING, workspace_name STRING, model_name STRING, table_count BIGINT, column_count BIGINT, measure_count BIGINT, relationship_count BIGINT, datasource_count BIGINT, analysis_run_id STRING, catalog_scan_id STRING, score_version BIGINT, security_schema_version BIGINT, security_scan_status STRING, security_fingerprint STRING, security_definition_json STRING, role_count BIGINT, rls_filter_count BIGINT, table_ols_count BIGINT, column_ols_count BIGINT, query_count BIGINT",
    "semantic_model_similarity_pairs": "model_id_a STRING, model_a STRING, workspace_a STRING, model_id_b STRING, model_b STRING, workspace_b STRING, same_model_name BOOLEAN, cross_workspace BOOLEAN, jaccard_tables DOUBLE, jaccard_columns DOUBLE, jaccard_measure_names DOUBLE, jaccard_relationships DOUBLE, jaccard_datasources DOUBLE, dax_embedding_cosine DOUBLE, composite_score DOUBLE, containment_score DOUBLE, containment_relationship STRING, model_a_in_model_b DOUBLE, model_b_in_model_a DOUBLE, tier STRING, schema_score DOUBLE, security_score DOUBLE, combined_score DOUBLE, score_mode STRING, security_comparison_status STRING, security_evidence_json STRING, security_fingerprint_a STRING, security_fingerprint_b STRING, catalog_scan_id_a STRING, catalog_scan_id_b STRING, analysis_run_id STRING, score_version BIGINT, power_query_similarity DOUBLE, power_query_status STRING, shared_query_count BIGINT",
    "semantic_model_duplicate_clusters": "cluster_id BIGINT, cluster_size BIGINT, model_id STRING, model STRING, workspace_name STRING, model_name STRING, analysis_run_id STRING, score_version BIGINT",
}

for table_name, frame in similarity_outputs.items():
    records = frame.to_dict("records")
    integer_fields = {field.split()[0] for field in similarity_schemas[table_name].split(",") if field.split()[1] == "BIGINT"}
    for record in records:
        for field, value in record.items():
            if pd.isna(value):
                record[field] = None
            elif field in integer_fields:
                record[field] = int(value)
    spark.createDataFrame(records, schema=similarity_schemas[table_name]).write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(table_name)
    print(f"Wrote {len(frame)} rows to {table_name}")

from datetime import datetime, timezone

_run_meta = [{
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "analysis_run_id": analysis_run_id,
    "catalog_scan_id": source_catalog_scan_id,
    "score_version": SCORE_VERSION,
    "security_schema_version": 1,
    "duplicate_threshold": float(DUPLICATE_THRESHOLD),
    "similar_threshold": float(SIMILAR_THRESHOLD),
    "containment_threshold": float(CONTAINMENT_THRESHOLD),
    "enable_blocking": bool(ENABLE_BLOCKING),
    "similarity_weights_json": security_json(SIMILARITY_WEIGHTS),
    "containment_weights_json": security_json(CONTAINMENT_WEIGHTS),
    "combined_weights_json": security_json(normalize_score_weights(COMBINED_WEIGHTS)),
    "security_weights_json": security_json(SECURITY_WEIGHTS),
    "security_role_weights_json": security_json(SECURITY_ROLE_WEIGHTS),
    "model_count": len(signatures_df),
    "pair_count": len(pairs_df),
    "duplicate_count": duplicate_count,
    "similar_count": similar_count,
    "unassessed_count": unassessed_count,
    "containment_count": containment_count,
    "cluster_count": cluster_number,
}]
run_schema = "generated_at STRING, analysis_run_id STRING, catalog_scan_id STRING, score_version BIGINT, security_schema_version BIGINT, duplicate_threshold DOUBLE, similar_threshold DOUBLE, containment_threshold DOUBLE, enable_blocking BOOLEAN, similarity_weights_json STRING, containment_weights_json STRING, combined_weights_json STRING, security_weights_json STRING, security_role_weights_json STRING, model_count BIGINT, pair_count BIGINT, duplicate_count BIGINT, similar_count BIGINT, unassessed_count BIGINT, containment_count BIGINT, cluster_count BIGINT"
spark.createDataFrame(_run_meta, schema=run_schema).write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("semantic_model_similarity_run")
print("Wrote versioned run metadata to semantic_model_similarity_run")


## Next steps

- Run **002_semantic_model_similarity_results** and use **Review**, **Groups**, **Compare**, and **Similarity map**. Three scores distinguish schema overlap, security-definition similarity, and the combined ranking.
- Possible duplicates follow the combined cutoff even when security differs. With the default weights, 100% schema and 0% security produce 95% combined and qualify at the inclusive 95% cutoff. Review the security warning before making consolidation decisions.
- Both models confirmed without roles use schema as combined and show security as not applicable. Missing, unreadable, partial, or inconsistent security leaves security and combined unavailable. Run **001 -> 002** after upgrading or changing model security.
- Schema coverage remains directional and excludes security. It does not establish permission containment, matching data, or replacement safety. Matching role definitions do not prove matching assignments or effective user access.
- Adjust `SIMILARITY_WEIGHTS`, `COMBINED_WEIGHTS`, `SECURITY_WEIGHTS`, `SECURITY_ROLE_WEIGHTS`, and review thresholds in Parameters, then rerun scoring. Weights and scan provenance are recorded with the results.
- `composite_score` remains the legacy schema-only value. New consumers should use `schema_score`, `security_score`, `combined_score`, `score_mode`, and `security_comparison_status`. The versioned outputs are saved to the same attached Lakehouse.